## Model Inference

In [58]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
).to("cuda")

model.eval()

print("Model loaded")
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Device:", next(model.parameters()).device)

Model loaded
Parameters: 494032768
Device: cuda:0


In [2]:
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

GPU memory allocated: 0.93 GB


In [ ]:
text = "The capital of Germany is"

tokens = tokenizer(text, return_tensors="pt") #return as PyTorch tensors

print("Token IDs:", tokens["input_ids"])
print("Tokens:", tokenizer.convert_ids_to_tokens(
    tokens["input_ids"][0]
))

Token IDs: tensor([[ 785, 6722,  315, 9856,  374]])
Tokens: ['The', 'Ġcapital', 'Ġof', 'ĠGermany', 'Ġis']


In [4]:
inputs = tokenizer(
    "The capital of Germany is",
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

print("Logits shape:", logits.shape)

Logits shape: torch.Size([1, 5, 151936])


In [5]:
next_token_logits = logits[0, -1, :]

probabilities = torch.softmax(
    next_token_logits.float(), dim=-1
)

top_probs, top_ids = torch.topk(probabilities, k=5)

for token_id, probability in zip(top_ids, top_probs):
    token = tokenizer.decode([token_id.item()])
    print(repr(token), round(probability.item(), 4))

' Berlin' 0.2493
' located' 0.0875
' __' 0.0828
' ______' 0.0766
' the' 0.0458


A correct prediction doesn't necessarily receive a high probability, and high token probability doesn't guarantee factual correctness

### Greedy Decoding

In [6]:
prompt = "The capital of Germany is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The capital of Germany is Berlin. It was founded in 1871, and it has been the capital since 1945. The city is located on


In [8]:
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
))

The capital of Germany is Berlin. It was founded in 1871, and it has been the capital since 1945. The city is located on


In [9]:
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
))

The capital of Germany is Berlin. It was founded in 1871, and it has been the capital since 1945. The city is located on


### Sampling

In [7]:
prompt = "The capital of Germany is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
))

The capital of Germany is located on the
A. Rhine River
B. Danube River
C. Upper Rhine River
D. Lower Rhine River




In [10]:
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
))

The capital of Germany is Berlin. It was founded by the Romans in 106 AD. The city has a population of more than two million and is located on the


In [11]:
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
))

The capital of Germany is ____
A. Berlin
B. Munich
C. Frankfurt
D. Hamburg
Answer:
B

In the following sentences, which one uses


### Reliability Experiment

In [16]:
prompt = "Explain why the sky appears blue."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

In [17]:
#Greedy Decoding
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
))

Explain why the sky appears blue. The sky appears blue because of the scattering of sunlight by tiny particles in the Earth's atmosphere, such as water vapor and nitrogen gas. When sunlight passes


In [18]:
#Sampling

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
))


Explain why the sky appears blue. The sky appears blue because it is a reflection of sunlight. When light enters the Earth's atmosphere, it passes through different layers and collides with other


As observed above, decoding parameters don't change what the model learned. They change how the outputs are selected from its learned probability distribution.
Now, we test whether increasing sampling temperature reduces factual reliability.

In [34]:
messages = [
    {
        "role": "user",
        "content": "Explain where is Tübingen located."


    }
        
]

In [36]:
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")


In [37]:
for temp in [0.2, 0.8, 1.5]:
    print(f"\nTemperature: {temp}")

    for run in range(3):
        with torch.no_grad():
            output = model.generate(
                input_ids=inputs,
                max_new_tokens=80,
                do_sample=True,
                temperature=temp,
                top_k=50,
                pad_token_id=tokenizer.eos_token_id
            )

        generated = output[0, inputs.shape[-1]:]

        print(
            f"Run {run+1}:",
            tokenizer.decode(
                generated,
                skip_special_tokens=True
            )
        )


Temperature: 0.2
Run 1: Tübingen is a city in the state of Baden-Württemberg, Germany. It is situated on the banks of the River Neckar and is known for its historical significance as the birthplace of Johann Sebastian Bach. The city has a rich cultural heritage and is home to several notable institutions, including the Bach Academy and the Bach Museum. Tübingen also plays an important
Run 2: Tübingen is a city in the state of Baden-Württemberg, Germany. It is situated on the banks of the River Neckar and is known for its historical significance and cultural heritage. Here are some key points about Tübingen:

1. Location: Tübingen is located in the eastern part of Baden-Württemberg, near the border
Run 3: Tübingen is a city in the state of Baden-Württemberg, Germany. It is situated on the banks of the River Neckar and is known for its historical significance as the birthplace of Johann Sebastian Bach. The city has a rich cultural heritage and is home to several notable institutions suc

| Temperature  | Coherence | Factual Reliability  | Observation |
|----------|-------|--------| --------|
| 0.2   | High  | Partially Correct     | Recurring factual errors, consistent explanation |
| 0.8   | Moderate  | Poor     | More variation and several fabricated explanations |
| 1.5   | Low  | Very Poor     | More 'creative' |


### Entropy

In [38]:
def inspect_prediction(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model(**inputs)

    # Scores for the next token
    logits = outputs.logits[0, -1, :].float()

    # Convert scores to probabilities
    probs = torch.softmax(logits, dim=-1)

    # Shannon entropy (in bits)
    entropy = -torch.sum(
        probs * torch.log2(probs.clamp_min(1e-12))
    )

    top_probs, top_ids = torch.topk(probs, k=5)

    print(f"\nPrompt: {prompt}")
    print(f"Entropy: {entropy.item():.3f} bits")

    for token_id, probability in zip(top_ids, top_probs):
        token = tokenizer.decode([token_id.item()])
        print(f"{repr(token):20} {probability.item():.4f}")

In [39]:
inspect_prediction("The capital of Germany is")


Prompt: The capital of Germany is
Entropy: 4.907 bits
' Berlin'            0.2493
' located'           0.0875
' __'                0.0828
' ______'            0.0766
' the'               0.0458


In [40]:
inspect_prediction("The capital of France is")


Prompt: The capital of France is
Entropy: 4.428 bits
' Paris'             0.3085
' ______'            0.1208
':\n'                0.0657
':\n\n'              0.0580
' __'                0.0481


In [43]:
inspect_prediction("The capital of the fictional country Zoravia is")


Prompt: The capital of the fictional country Zoravia is
Entropy: 0.640 bits
' located'           0.9449
' at'                0.0139
' named'             0.0042
' surrounded'        0.0041
' standing'          0.0035


Low entropy: The model strongly favors a few tokens.
High entropy: The model distributes probability across many possible tokens.

Low next-token entropy can reflect confidence in linguistic structure rather than confidence in factual knowledge, as per the results. So we try to remove a generic word like "located" and force the model to give out a name. For example, the model is not strongly identifying Zoravia's capital but it is selecting a grammatically plausible continuation.


In [44]:
inspect_prediction(
    "Germany's capital city is called"
)


Prompt: Germany's capital city is called
Entropy: 4.170 bits
' Berlin'            0.3711
' the'               0.1303
' Hamburg'           0.0754
' "'                 0.0362
' ____'              0.0302


In [45]:

inspect_prediction(
    "France's capital city is called"
)



Prompt: France's capital city is called
Entropy: 4.097 bits
' Paris'             0.3681
' the'               0.1752
' "'                 0.0650
' ____'              0.0385
':\n'                0.0345


In [46]:
inspect_prediction(
    "Zoravia's capital city is called"
)


Prompt: Zoravia's capital city is called
Entropy: 8.988 bits
' "'                 0.1073
' the'               0.0755
' Prague'            0.0236
' Z'                 0.0221
' K'                 0.0162


Changing "is" to "is called" dramatically changed Zoravia's next-token distribution and increased Entropy, which supports the hypothesis that the measurement depends on how we formulate the question.

### Hidden States

In [47]:
prompt = "The capital of Germany is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model(
        **inputs,
        output_hidden_states=True
    )

hidden_states = outputs.hidden_states

print("Number of hidden-state tensors:", len(hidden_states))
print("First tensor shape:", hidden_states[0].shape)
print("Last tensor shape:", hidden_states[-1].shape)

Number of hidden-state tensors: 25
First tensor shape: torch.Size([1, 5, 896])
Last tensor shape: torch.Size([1, 5, 896])


In [48]:
import torch.nn.functional as F

initial = hidden_states[0][0, -1, :].float()

for layer_idx in [1, 6, 12, 18, 24]:
    current = hidden_states[layer_idx][0, -1, :].float()

    similarity = F.cosine_similarity(
        initial.unsqueeze(0),
        current.unsqueeze(0)
    ).item()

    print(f"Layer {layer_idx}: {similarity:.4f}")

Layer 1: 0.0071
Layer 6: -0.0072
Layer 12: 0.0153
Layer 18: -0.0057
Layer 24: 0.0765


The hidden-state vectors are nearly orthogonal to the initial representation, as the cosine similarity values are very close to zero. This does not mean that the model has forgotten or lost the meaning of the original token. There are other factors that can affect these comparisons. Cosine similarity can describe changes in hidden representations, but it doesn't establish what information the model encodes or which computations cause its predictions.

In [49]:
for i in range(1, len(hidden_states)):
    previous = hidden_states[i-1][0, -1, :].float()
    current = hidden_states[i][0, -1, :].float()

    similarity = F.cosine_similarity(
        previous, current, dim=0
    ).item()

    print(f"Layer {i-1} → {i}: {similarity:.4f}")

Layer 0 → 1: 0.0071
Layer 1 → 2: 0.8598
Layer 2 → 3: 0.9238
Layer 3 → 4: 0.9086
Layer 4 → 5: 0.8840
Layer 5 → 6: 0.8880
Layer 6 → 7: 0.9071
Layer 7 → 8: 0.8970
Layer 8 → 9: 0.9074
Layer 9 → 10: 0.9413
Layer 10 → 11: 0.9347
Layer 11 → 12: 0.9325
Layer 12 → 13: 0.9433
Layer 13 → 14: 0.9500
Layer 14 → 15: 0.9524
Layer 15 → 16: 0.9144
Layer 16 → 17: 0.9232
Layer 17 → 18: 0.9566
Layer 18 → 19: 0.9530
Layer 19 → 20: 0.9551
Layer 20 → 21: 0.9188
Layer 21 → 22: 0.9239
Layer 22 → 23: 0.9587
Layer 23 → 24: 0.3585


Layer 0 -> 1: The representation changes direction substantially at the first transformer layer, as the cosine values is small

Layer 1 -> 23 Within the range 0.86 - 0.96. The layer updates the existing representation rather than constructing an entirely unrelated vector. Although the changes might be small, they still count as it might strongly affect the probability of a particular output token.


Layer 23 -> 24: 0.36 approximately, but it is important to consider that the last transition includes the final transformer layer and the final normalization. In Hugging Face's Qwen2 implementation, the last returned hidden state is typically taken after the model's final RMS normalization.

**Large representation changes do not necessarily imply large changes in predictions, and small representation changes do not necessarily imply insignificant computations.**

In [50]:
prompts = [
    "The capital of Germany is",
    "The capital of France is",
    "Zoravia's capital city is called"
]

In [51]:
for prompt in prompts:
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True
        )

    states = outputs.hidden_states

    similarities = []

    for i in range(1, len(states)):
        sim = F.cosine_similarity(
            states[i-1][0, -1, :].float(),
            states[i][0, -1, :].float(),
            dim=0
        ).item()

        similarities.append(sim)

    print(f"\n{prompt}")
    print("First transition:", round(similarities[0], 4))
    print("Middle transition:", round(similarities[11], 4))
    print("Final transition:", round(similarities[-1], 4))


The capital of Germany is
First transition: 0.0071
Middle transition: 0.9325
Final transition: 0.3585

The capital of France is
First transition: 0.0083
Middle transition: 0.9348
Final transition: 0.37

Zoravia's capital city is called
First transition: 0.0731
Middle transition: 0.9164
Final transition: 0.2855


First transition: All three show a substantial directional change.

Middle transition: All three retain high cosine similarity, suggesting relatively small directional updates.

Final transition: All three show another substantial directional change.

The overall pattern is similar, although Zoravia differs somewhat in the individual values.

### Logit Lens

In [52]:
prompt = "The capital of Germany is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

Logit-lens predictions are diagnostic approximations, not necessarily the model's actual intermediate beliefs. Qwen uses RMSNorm before its final output projection and the intermediate hidden states haven't passed through them so we apply it before the language model head.

In [53]:
with torch.no_grad():
    outputs = model(
        **inputs,
        output_hidden_states=True
    )

    for layer_idx in [1, 6, 12, 18, 24]:
        hidden = outputs.hidden_states[layer_idx]

        # Apply the model's final normalization
        if layer_idx < 24:
            hidden = model.model.norm(hidden)

        # Project into vocabulary space
        logits = model.lm_head(hidden)

        # Most likely next token
        next_token_id = logits[0, -1, :].argmax().item()

        token = tokenizer.decode([next_token_id])

        print(f"Layer {layer_idx}: {repr(token)}")

Layer 1: '\';";\n'
Layer 6: '\';";\n'
Layer 12: '\';";\n'
Layer 18: '____'
Layer 24: ' Berlin'


The intermediate representations do not yet align with the final output projection in a way that makes " Berlin" the highest-scoring token until the final layer. We cannot conclude that the model only learns or retrieves Berlin at layer 24. The information might already exist in earlier representations without being directly decodable by the standard output head. As stated before, this is a limitation of logit-lens.

In [54]:
target_id = tokenizer.encode(
    " Berlin",
    add_special_tokens=False
)[0]

print("Target token:", tokenizer.decode([target_id]))

Target token:  Berlin


Verify if "Berlin" is one token

In [56]:
ids = tokenizer.encode(
    " Berlin",
    add_special_tokens=False
)

print("Token IDs:", ids)
print("Number of tokens:", len(ids))
print("Tokens:", tokenizer.convert_ids_to_tokens(ids))

Token IDs: [19846]
Number of tokens: 1
Tokens: ['ĠBerlin']


In [55]:
with torch.no_grad():
    for layer_idx in [1, 6, 12, 18, 24]:
        hidden = outputs.hidden_states[layer_idx]

        if layer_idx < 24:
            hidden = model.model.norm(hidden)

        logits = model.lm_head(hidden)
        probs = torch.softmax(
            logits[0, -1, :].float(),
            dim=-1
        )

        target_prob = probs[target_id].item()
        target_rank = (
            (logits[0, -1, :] > logits[0, -1, target_id])
            .sum().item() + 1
        )

        print(
            f"Layer {layer_idx}: "
            f"P(Berlin)={target_prob:.6f}, "
            f"Rank={target_rank}"
        )

Layer 1: P(Berlin)=0.000003, Rank=13989
Layer 6: P(Berlin)=0.000000, Rank=131930
Layer 12: P(Berlin)=0.000000, Rank=120519
Layer 18: P(Berlin)=0.000000, Rank=49041
Layer 24: P(Berlin)=0.249269, Rank=1


We can see that the rank of Berlin actually becomes worse from Layer 1 -> Layer 6 and improves later. This experiment cannot determine if information about this particular token is present in an intermediate hidden stage but it suggests that it cannot effectively identify the token until the final representation.